# Segmentación no supervisada — ¿qué *tipos de día* existen? 🧩

Complemento del EDA (`01_eda.ipynb`) y del modelo supervisado: aquí **no usamos la etiqueta**
para entrenar, sino que dejamos que **K-Means** descubra por sí solo los patrones de demanda,
validando la elección del número de grupos con el **coeficiente de silhouette** y visualizando
la estructura con **PCA**.

**Idea central:** representamos cada día como su *perfil horario* (cómo se reparte la demanda
en las 24 horas) y agrupamos días con perfiles parecidos. Si los grupos que emergen coinciden
con conceptos reales (día laboral, fin de semana, mal tiempo), eso **valida las variables**
que luego usa el modelo supervisado.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

from bikeshare.etl.load import load_processed

TEMPLATE = "plotly_white"
# Paleta categórica del proyecto (validada para accesibilidad y daltonismo)
C_VERDE, C_VIOLETA, C_NARANJA, C_AZUL = "#2b8a3e", "#7048e8", "#e8590c", "#1971c2"

df = load_processed()
print(f"Dataset procesado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
df["date"] = df["datetime"].dt.date  # clave de agrupación día a día

Dataset procesado: 17,158 filas × 42 columnas


## 1. Matriz de perfiles diarios (día × 24 horas)

Cada fila es un día y cada columna la demanda media de esa hora. Dos decisiones importantes:

- **Filtramos los días incompletos** (con menos de 24 horas registradas): un hueco
  deformaría el perfil y K-Means lo confundiría con un patrón real.
- **Normalizamos cada día por su total**: nos interesa la **forma** del perfil
  (*¿a qué horas se concentra la demanda?*), no el volumen. Sin esto, K-Means agruparía
  por cantidad total (días de 2011 vs 2012) en lugar de por comportamiento.

In [2]:
pivot = df.pivot_table(index="date", columns="hour", values="cnt", aggfunc="mean")
complete = pivot.dropna()  # solo días con las 24 horas

# Normalización: cada fila pasa a sumar 1 (distribución horaria del día)
profiles = complete.div(complete.sum(axis=1), axis=0)
X = profiles.values

print(f"Días totales: {len(pivot)} → días completos usados: {len(complete)}")
profiles.head(3).round(3)

Días totales: 729 → días completos usados: 612


hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
date,,,,,,,,,,,,,,,,,,,,,
2011-01-09,0.030,0.015,0.013,0.005,0.001,0.001,0.001,0.007,0.012,0.023,...,0.088,0.100,0.112,0.075,0.058,0.050,0.046,0.024,0.018,0.007
2011-01-10,0.004,0.001,0.002,0.001,0.002,0.002,0.023,0.058,0.142,0.071,...,0.036,0.034,0.056,0.135,0.117,0.072,0.056,0.029,0.018,0.014
2011-01-16,0.032,0.019,0.013,0.012,0.001,0.002,0.001,0.002,0.015,0.027,...,0.076,0.094,0.082,0.087,0.056,0.051,0.047,0.023,0.017,0.015


## 2. ¿Cuántos grupos hay? Método del codo + silhouette

En lugar de fijar *k* a ojo, usamos **dos criterios complementarios**:

- **Método del codo**: la inercia (suma de distancias² al centroide) siempre baja al
  aumentar k; buscamos el punto donde deja de bajar "en picada". Importante graficar
  **desde k = 1** — si se omite, el codo aparenta estar más a la derecha.
- **Coeficiente de silhouette**: cohesión interna vs separación entre clusters
  (más alto = mejor). Solo está definido desde k = 2.

In [3]:
ks_elbow = list(range(1, 9))
inertias = []
scores = {}
for k in ks_elbow:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    if k >= 2:
        scores[k] = silhouette_score(X, labels)

elbow = pd.DataFrame({"k": ks_elbow, "inercia": inertias})
sil = pd.DataFrame({"k": list(scores), "silhouette": list(scores.values())})
best_k = int(sil.loc[sil["silhouette"].idxmax(), "k"])

fig = px.line(elbow, x="k", y="inercia", markers=True, template=TEMPLATE,
              title="Método del codo — inercia según k",
              labels={"k": "Número de clusters (k)", "inercia": "Inercia (WCSS)"})
fig.update_traces(line_color=C_VERDE, marker=dict(size=9))
fig.add_annotation(x=2, y=elbow.loc[elbow["k"] == 2, "inercia"].iloc[0],
                   text="codo: la caída 1→2 es −69%;<br>de ahí en adelante < −15%",
                   showarrow=True, arrowhead=2, ax=60, ay=-45)
fig.update_layout(margin=dict(l=40, r=20, t=50, b=40))
fig.show()

fig = px.line(sil, x="k", y="silhouette", markers=True, template=TEMPLATE,
              title="Coeficiente de silhouette según k",
              labels={"k": "Número de clusters (k)", "silhouette": "Silhouette promedio"})
fig.update_traces(line_color=C_VERDE, marker=dict(size=9))
fig.add_annotation(x=best_k, y=sil["silhouette"].max(),
                   text=f"mejor k = {best_k}", showarrow=True, arrowhead=2, ay=-30)
fig.update_layout(margin=dict(l=40, r=20, t=50, b=40))
fig.show()
sil.round(3)

,k,silhouette
0,2,0.625
1,3,0.382
2,4,0.360
3,5,0.241
4,6,0.179
5,7,0.236
6,8,0.204


**Lectura — los dos criterios coinciden en k = 2:**

- **Codo:** la inercia cae **69%** al pasar de 1 a 2 clusters y después las caídas son
  suaves y decrecientes (14%, 10%, 9%…), sin ningún otro quiebre. Formalmente, k = 2 es
  también el punto más alejado de la recta entre los extremos (criterio *kneedle*).
- **Silhouette:** máximo claro en **k = 2** (≈ 0.62, muy por encima de 0.38 de k = 3).

La estructura *dominante* de la demanda son **dos regímenes**. Eso no significa que no
haya nada más: en la sección 5 aumentamos k para explorar la subestructura (y aparece un
grupo nuevo con historia propia).

## 3. K-Means con k = 2: los dos regímenes de demanda

Ajustamos K-Means y, como sus etiquetas (0/1) son arbitrarias, **nombramos cada cluster
según su composición real** cruzando con variables de calendario y clima.

In [4]:
km2 = KMeans(n_clusters=2, random_state=42, n_init=10)
labels2 = pd.Series(km2.fit_predict(X), index=profiles.index, name="cluster")

# Metadatos por día para interpretar los clusters
meta = df.groupby("date").agg(
    workingday=("workingday", "max"),
    is_holiday=("is_holiday", "max"),
    temp_media=("temperature_2m", "mean"),
    precip_total=("precipitation", "sum"),
    demanda_total=("cnt", "sum"),
).join(labels2, how="inner")

pct_lab = meta.groupby("cluster")["workingday"].mean()
nombres2 = {c: ("Día laboral" if p > 0.5 else "Fin de semana / feriado")
            for c, p in pct_lab.items()}
colores2 = {"Día laboral": C_VERDE, "Fin de semana / feriado": C_VIOLETA}
meta["tipo"] = meta["cluster"].map(nombres2)

fig = go.Figure()
for nombre in ["Día laboral", "Fin de semana / feriado"]:
    perfil = profiles.loc[meta.index[meta["tipo"] == nombre]].mean() * 100
    fig.add_scatter(x=list(perfil.index), y=perfil.values, name=nombre,
                    line=dict(color=colores2[nombre], width=2))
    fig.add_annotation(x=int(perfil.idxmax()), y=perfil.max(), text=nombre,
                       font=dict(color=colores2[nombre], size=12),
                       showarrow=False, yshift=14)
fig.update_layout(template=TEMPLATE, title="Perfil horario promedio de cada cluster (k=2)",
                  xaxis_title="Hora del día", yaxis_title="% de la demanda diaria",
                  legend=dict(orientation="h"), margin=dict(l=40, r=20, t=50, b=40))
fig.show()

In [5]:
resumen2 = meta.groupby("tipo").agg(
    dias=("cluster", "size"),
    pct_dia_laboral=("workingday", "mean"),
    pct_feriado=("is_holiday", "mean"),
    temp_media=("temp_media", "mean"),
    precip_total_media=("precip_total", "mean"),
    demanda_diaria_media=("demanda_total", "mean"),
).round(2)
resumen2

,dias,pct_dia_laboral,pct_feriado,temp_media,precip_total_media,demanda_diaria_media
tipo,,,,,,
Día laboral,405,0.99,0.00,16.89,2.29,5071.96
Fin de semana / feriado,207,0.03,0.07,14.69,2.44,4636.06


**Lectura:** K-Means, **sin conocer el calendario**, redescubrió la división laboral / no laboral
(99% vs 3% de días laborales por cluster):

- **Día laboral** — doble peak de *commuting* (≈ 8:00 y ≈ 17–18:00).
- **Fin de semana / feriado** — curva suave concentrada al mediodía (uso recreativo).

Nótese que la **demanda total diaria es parecida** en ambos grupos (~5.100 vs ~4.600):
lo que cambia radicalmente es **cuándo** se usa la bicicleta, no cuánto. Por eso
clusterizamos la *forma* del perfil y no el volumen — y por eso `workingday` y la hora
son de las variables más importantes del modelo supervisado.

## 4. PCA: la estructura vista en 2 dimensiones

Cada día vive en un espacio de 24 dimensiones (una por hora). **PCA** lo proyecta a 2
componentes principales para *ver* si los clusters están realmente separados.

In [6]:
pca = PCA(n_components=5, random_state=42)
coords = pca.fit_transform(X)
var1, var2 = pca.explained_variance_ratio_[:2] * 100

pca_df = pd.DataFrame({
    "PC1": coords[:, 0], "PC2": coords[:, 1],
    "tipo": meta.loc[profiles.index, "tipo"].values,
    "fecha": [str(d) for d in profiles.index],
})
fig = px.scatter(pca_df, x="PC1", y="PC2", color="tipo", color_discrete_map=colores2,
                 opacity=0.65, hover_data=["fecha"], template=TEMPLATE,
                 title="Días proyectados en las 2 primeras componentes principales",
                 labels={"PC1": f"PC1 ({var1:.0f}% de la varianza)",
                         "PC2": f"PC2 ({var2:.0f}% de la varianza)"})
fig.update_traces(marker=dict(size=7))
fig.update_layout(legend=dict(orientation="h"), margin=dict(l=40, r=20, t=50, b=40))
fig.show()

In [7]:
var_df = pd.DataFrame({
    "componente": [f"PC{i + 1}" for i in range(5)],
    "varianza": pca.explained_variance_ratio_ * 100,
})
fig = px.bar(var_df, x="componente", y="varianza", template=TEMPLATE,
             title="Varianza explicada por componente (PCA)",
             labels={"componente": "", "varianza": "% de varianza"})
fig.update_traces(marker_color=C_VERDE)
fig.update_layout(margin=dict(l=40, r=20, t=50, b=40))
fig.show()
print(f"Las 2 primeras componentes explican el {var_df['varianza'].iloc[:2].sum():.0f}% "
      "de la varianza")

Las 2 primeras componentes explican el 86% de la varianza


**Lectura:** los dos grupos se separan **a lo largo de PC1**, que por sí sola concentra
~76% de la varianza (86% junto a PC2). Es decir, la diferencia laboral / no laboral **es**
la dirección principal de variación de los perfiles: la estructura de la demanda es de
baja dimensión y muy aprendible — coherente con el R² = 0.96 del modelo supervisado.

## 5. Más granularidad: k = 4 revela los días de lluvia

Silhouette elige k = 2 como estructura *dominante*, pero aumentar k permite explorar
subestructura. Con **k = 4** los días laborales se subdividen — y aparece un grupo pequeño
muy interesante.

In [8]:
km4 = KMeans(n_clusters=4, random_state=42, n_init=10)
meta["cluster4"] = km4.fit_predict(X)

# Nombramos por composición: el laboral más lluvioso, luego cálido vs frío
stats4 = meta.groupby("cluster4").agg(pct_lab=("workingday", "mean"),
                                      precip=("precip_total", "mean"),
                                      temp=("temp_media", "mean"))
nombres4 = {}
laborales = stats4[stats4["pct_lab"] > 0.5]
nombres4[laborales["precip"].idxmax()] = "Laboral lluvioso"
resto = laborales.drop(laborales["precip"].idxmax())
nombres4[resto["temp"].idxmax()] = "Laboral cálido"
nombres4[resto["temp"].idxmin()] = "Laboral frío"
for c in stats4.index:
    nombres4.setdefault(c, "Fin de semana / feriado")
meta["tipo4"] = meta["cluster4"].map(nombres4)

colores4 = {"Laboral cálido": C_VERDE, "Laboral frío": C_AZUL,
            "Laboral lluvioso": C_NARANJA, "Fin de semana / feriado": C_VIOLETA}

fig = go.Figure()
for nombre, color in colores4.items():
    perfil = profiles.loc[meta.index[meta["tipo4"] == nombre]].mean() * 100
    fig.add_scatter(x=list(perfil.index), y=perfil.values, name=nombre,
                    line=dict(color=color, width=2))
fig.update_layout(template=TEMPLATE, title="Perfil horario promedio por cluster (k=4)",
                  xaxis_title="Hora del día", yaxis_title="% de la demanda diaria",
                  legend=dict(orientation="h"), margin=dict(l=40, r=20, t=50, b=40))
fig.show()

In [9]:
resumen4 = meta.groupby("tipo4").agg(
    dias=("cluster4", "size"),
    pct_dia_laboral=("workingday", "mean"),
    temp_media=("temp_media", "mean"),
    precip_total_media=("precip_total", "mean"),
    demanda_diaria_media=("demanda_total", "mean"),
).round(2)
resumen4

,dias,pct_dia_laboral,temp_media,precip_total_media,demanda_diaria_media
tipo4,,,,,
Fin de semana / feriado,204,0.03,14.75,2.41,4634.28
Laboral cálido,276,1.00,18.51,1.76,5296.80
Laboral frío,105,0.96,13.43,1.74,5005.90
Laboral lluvioso,27,1.00,13.09,10.19,2995.33


**Lectura:** el cluster **"Laboral lluvioso"** (27 días) tiene una precipitación media
~5× mayor que el resto y una demanda diaria de ~3.000 bicicletas, un **~43% menos** que un
día laboral típico (~5.300). El mal tiempo no cambia *la forma* del día (sigue habiendo
doble peak: la gente igual va a trabajar) pero **deprime el volumen** — exactamente el rol
secundario-pero-real que el clima tiene en la importancia de variables del XGBoost.

## Conclusiones

1. **K-Means + silhouette** (k = 2) redescubre, sin etiquetas, la división
   **laboral / fin de semana**: los dos regímenes de uso (commuting vs recreativo).
2. **PCA** confirma que esa división es la dirección principal de variación
   (PC1 ≈ 76% de la varianza; 86% con 2 componentes): la estructura es de baja dimensión.
3. Con **k = 4** emerge el cluster de **días laborales lluviosos** (demanda −43%),
   cuantificando el efecto del clima que motivó integrar Open-Meteo en el ETL.
4. **Conexión con el modelo supervisado:** los grupos que el clustering descubre solo
   (día laboral, hora del día, clima) son precisamente las variables con mayor importancia
   en el XGBoost. Lo no supervisado **valida** el diseño de features de lo supervisado:
   ambos enfoques cuentan la misma historia desde ángulos distintos.